In this notebook, we estimate occupied minutes per hour using room sensor data. First, we calculate the minimum, maximum, and median for four sensors, creating 12 features. We split the data by random split into training, validation, and test sets. We use linear regression as our baseline and compare it with random forest using mean absolute error (MAE) and R² .



## 1 Load and prepare the data
Load the CSV, convert dates, and sort readings by time

In [ ]:
import pandas as pd

In [ ]:
df=pd.read_csv("Occupancy.csv")

In [ ]:
data = df.copy()
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values("date")

## 2 Create hourly features and target
Summarize each sensor by hour and estimate occupied minutes from the occupied proportion. Keep hours with 59–61 readings.

In [ ]:
# Four sensors × three summaries = 12 features
sensors = ["Temperature", "Humidity", "CO2", "Light"]
#group readings by hours
hour_groups = data.set_index("date").resample("h")
#caculate min reading, max reading, median reading for each sensor
X_hourly = hour_groups[sensors].agg(["min", "max", "median"])


In [ ]:
# Simplify column names, for example Temperature_min
X_hourly.columns = [
    f"{sensor}_{summary}"
    for sensor, summary in X_hourly.columns
]

In [ ]:
# Summarize occupancy and coverage for each hour
hourly_target = hour_groups["Occupancy"].agg(
    occupied_share="mean",
    readings="count"
)

# Estimate occupied minutes
hourly_target["occupied_minutes"] = (
    hourly_target["occupied_share"] * 60
)

hourly = X_hourly.join(hourly_target)

# For this dataset, keep hours with approximately full coverage
hourly = hourly[
    hourly["readings"].between(59, 61)
].copy()

display(hourly.head())

,Temperature_min,Temperature_max,Temperature_median,Humidity_min,Humidity_max,Humidity_median,CO2_min,CO2_max,CO2_median,Light_min,Light_max,Light_median,occupied_share,readings,occupied_minutes
date,,,,,,,,,,,,,,,
2015-02-02 15:00:00,23.00,23.60,23.290000,27.5560,29.075,28.5000,1026.25,1176.166667,1105.083333,429.0,480.142857,456.15,1.000000,60,60.000000
2015-02-02 16:00:00,22.60,23.00,22.700000,25.1400,28.200,26.7000,847.40,1091.750000,976.200000,428.2,444.000000,433.50,1.000000,61,60.000000
2015-02-02 17:00:00,22.39,22.60,22.540000,24.7000,25.200,25.0000,772.25,854.750000,822.500000,419.0,433.000000,428.25,0.610169,59,36.610169
2015-02-02 18:00:00,21.50,22.39,22.033333,24.2000,25.245,24.6125,638.00,809.000000,716.300000,0.0,419.000000,0.00,0.083333,60,5.000000
2015-02-02 19:00:00,21.10,21.50,21.254000,23.4725,24.200,23.8500,540.75,636.333333,585.000000,0.0,0.000000,0.00,0.000000,61,0.000000


## 3 Split the data


In [ ]:
from sklearn.model_selection import train_test_split

# Split
train_hourly, remaining_hourly = train_test_split(
    hourly, test_size=0.40, random_state=42
)
val_hourly, test_hourly = train_test_split(
    remaining_hourly, test_size=0.50, random_state=42
)

features = X_hourly.columns.tolist()

X_train_hourly = train_hourly[features]
y_train_hourly = train_hourly["occupied_minutes"]

X_val_hourly = val_hourly[features]
y_val_hourly = val_hourly["occupied_minutes"]


## 4 Train linear regression

Fit scaling and linear regression on Train, then calculate Validation MAE and R²

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error,r2_score

regression_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LinearRegression())
])

regression_pipeline.fit(X_train_hourly, y_train_hourly)

predictions = regression_pipeline.predict(X_val_hourly)

print(
    "Validation MAE:",
    mean_absolute_error(y_val_hourly, predictions)
)

print("Validation R²:", r2_score(y_val_hourly, predictions))

Validation MAE: 5.2870829954454805
Validation R²: 0.8823487666937061


## 5 Inspect validation results
Compare recorded estimates with predictions

In [ ]:
results = pd.DataFrame({
    "Recorded occupied minutes": y_val_hourly,
    "Predicted occupied minutes": predictions
})

display(results.head())

,Recorded occupied minutes,Predicted occupied minutes
date,,
2015-02-06 05:00:00,0.000000,-0.858646
2015-02-12 07:00:00,0.000000,5.468780
2015-02-15 19:00:00,0.000000,1.503877
2015-02-13 17:00:00,57.966102,47.349342
2015-02-02 22:00:00,0.000000,-0.045869


## 6 Train random forest
Fit random forest on the same training features and predict Validation.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
#scaling is not required
forest_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

#the same original 12 features
forest_model.fit(X_train_hourly, y_train_hourly)

forest_predictions = forest_model.predict(X_val_hourly)

## 7 Compare the models
Compare validation errors and predictions for linear regression and random forest.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Predictions from existing linear regression pipeline
linear_predictions = regression_pipeline.predict(X_val_hourly)

occupied = y_val_hourly > 0

comparison = pd.DataFrame({
    "Model": ["Linear regression — baseline", "Random forest"],
    "Validation MAE": [
        mean_absolute_error(y_val_hourly, linear_predictions),
        mean_absolute_error(y_val_hourly, forest_predictions)
    ],
     "Validation R²": [
        r2_score(y_val_hourly, linear_predictions),
        r2_score(y_val_hourly, forest_predictions)
     ],
    "MAE for occupied hours": [
        mean_absolute_error(
            y_val_hourly[occupied],
            linear_predictions[occupied]
        ),
        mean_absolute_error(
            y_val_hourly[occupied],
            forest_predictions[occupied]
        )
    ]
})

display(comparison.round(2))

,Model,Validation MAE,Validation R²,MAE for occupied hours
0,Linear regression — baseline,5.29,0.88,8.90
1,Random forest,1.87,0.96,4.91


In [ ]:
results_comparison = pd.DataFrame({
    "Target occupied minutes": y_val_hourly,
    "Linear regression": linear_predictions,
    "Random forest": forest_predictions
})

display(results_comparison.round(2))

,Target occupied minutes,Linear regression,Random forest
date,,,
2015-02-06 05:00:00,0.00,-0.86,0.00
2015-02-12 07:00:00,0.00,5.47,0.00
2015-02-15 19:00:00,0.00,1.50,0.00
2015-02-13 17:00:00,57.97,47.35,54.47
2015-02-02 22:00:00,0.00,-0.05,0.00
...,...,...,...
2015-02-08 16:00:00,0.00,-0.13,0.00
2015-02-09 18:00:00,5.00,7.63,19.56
2015-02-07 12:00:00,0.00,29.06,2.83
